# Previsão de afluências indevidas

Estima a afluência indevida de agora com a altura da água e a chuva. O d+5, sem altura futura, é o caudal médio de cada hora do relógio. Quando chega uma altura nova, corre-se outra vez: a linha reancora e a previsão anterior fica tracejada.

A linha azul cheia é o caudal medido do CSV que o estágio 03 lê (a partir de julho de 2026), somado em m³/dia. O nowcast e a previsão a vermelho continuam a ser a afluência indevida. No treino só entram horas com altura dentro do coletor. Blocos de caudal ≤ 0 com 6 h ou mais contam como sensor parado e saem; uma hora em que metade ou mais das leituras são zero também sai.

Quando os primeiros dias são fragmentos, o teste são os últimos três dias e o treino só os dias anteriores com pelo menos 20 horas. As constantes de esgoto e de indireta desse teste saem só do bloco de treino. O modelo aprende a afluência, `caudal − esgoto − indireta`, com a chuva das 3 horas anteriores, a das 6 horas anteriores e a hora do dia.

A simulação no fim da série usa a chuva que realmente caiu, como se a previsão meteorológica fosse perfeita. A linha dos 5 dias é a que erra menos nos dias de chuva: a correção pela chuva, ou repetir o nowcast de hoje quando a chuva não ajuda.


In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
predict_dir = cwd if (cwd / "previsao.py").exists() else cwd / "predict inflow"
project_dir = predict_dir.parent
for path in (project_dir, predict_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import importlib

import previsao
importlib.reload(previsao)

from pipeline_config import STATIONS
from previsao import NOMES, run_station


In [2]:
import pandas as pd
import plotly.graph_objects as go

def metrics_text(erro, score, titulo: str) -> str:
    """As métricas do run_station em linhas de texto para uma anotação plotly."""
    linhas = [f"<b>{titulo}</b>"]
    for nome, tabela, coluna, fmt in (
        ("R² caudal total", score, "caudal_total", ".3f"),
        ("Erro médio abs. caudal total (m³/h)", erro, "caudal_total", ".2f"),
        ("R² afluência direta", score, "afluencia_direta", ".3f"),
        ("Erro médio abs. afluência direta (m³/h)", erro, "afluencia_direta", ".2f"),
    ):
        linhas.append(f"{nome}: " + " · ".join(f"{NOMES[s]} {tabela.loc[s, coluna]:{fmt}}" for s in NOMES))
    return "<br>".join(linhas)


def add_metrics(fig: go.Figure, text: str) -> None:
    """Escreve as métricas por baixo dos eixos, no canto inferior esquerdo."""
    fig.add_annotation(
        text=text, xref="paper", yref="paper", x=0, y=-0.06, xanchor="left", yanchor="top",
        showarrow=False, align="left", bgcolor="white", bordercolor="gray", borderwidth=1, font=dict(size=12),
    )
    fig.update_layout(margin=dict(b=170), height=(fig.layout.height or 600) + 130)


def test_period(fig: go.Figure) -> tuple:
    """Primeira e última hora do teste e número de horas, lidas da linha do caudal medido."""
    medido = next(trace for trace in fig.data if str(trace.legendgroup).startswith("medido"))
    horas = pd.to_datetime(list(medido.x))
    return horas.min(), horas.max(), len(horas)


In [3]:
for estacao in STATIONS:
    result = run_station(estacao)
    label = STATIONS[estacao]["folder"]
    print(result["veredito"])

    inicio, fim, horas = test_period(result["figura"])
    titulo = f"Teste {inicio:%d/%m} → {fim:%d/%m} ({horas} h)"
    print(f"=== {label} — métricas no teste ({horas} horas) ===")
    print("erro médio absoluto (m³/h):")
    print(result["erro"].rename(index=NOMES).round(2).to_string())
    print("R²:")
    print(result["score"].rename(index=NOMES).round(3).to_string())

    add_metrics(result["figura"], metrics_text(result["erro"], result["score"], titulo))
    result["figura"].write_html(result["html"])
    print(result["html"])
    result["figura"].show()


Ameal: 138 horas de treino em 6 dias, 302 horas na série
  componentes do treino (N=1, 1 madrugadas): indireta 2.70 m³/h, esgoto 1.84 m³/h
  modelos sem altura: alvo = caudal − esgoto − indireta, constantes só do treino
  parâmetros (validação cruzada no treino, erro da afluência em m³/h):
    previsão (com altura): {'learning_rate': 0.1, 'max_depth': 3, 'max_iter': 100, 'min_samples_leaf': 5} → 0.68
    previsão (só altura): {'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'min_samples_leaf': 5} → 0.85
    previsão (só chuva): {'learning_rate': 0.03, 'max_depth': None, 'max_iter': 100, 'min_samples_leaf': 5} → 0.45
  teste 2026-08-31 → 2026-09-02 (62 horas), erro médio absoluto em m³/h:
                       caudal_total  afluencia_direta
previsão (com altura)          0.32              0.16
previsão (só altura)           0.50              0.17
previsão (só chuva)            0.87              0.36
  score R² no teste:
                       caudal_total  afluencia_direta
prev

Antanhol: 703 horas de treino em 30 dias, 1006 horas na série
  componentes do treino (N=7, 1 madrugadas): indireta 1.26 m³/h, esgoto 22.58 m³/h
  modelos sem altura: alvo = caudal − esgoto − indireta, constantes só do treino
  parâmetros (validação cruzada no treino, erro da afluência em m³/h):
    previsão (com altura): {'learning_rate': 0.03, 'max_depth': None, 'max_iter': 300, 'min_samples_leaf': 5} → 0.50
    previsão (só altura): {'learning_rate': 0.03, 'max_depth': None, 'max_iter': 300, 'min_samples_leaf': 5} → 0.46
    previsão (só chuva): {'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'min_samples_leaf': 5} → 5.49
  teste 2026-08-21 → 2026-09-02 (303 horas), erro médio absoluto em m³/h:
                       caudal_total  afluencia_direta
previsão (com altura)          0.51              0.43
previsão (só altura)           0.46              0.38
previsão (só chuva)           15.55              9.72
  score R² no teste:
                       caudal_total  afluencia_